02 - Split Data

Goal of this notebook:
- load the validated dataset
- sort it chronologically
- create train / validation / test splits
- make sure the split is time-safe
- save split parquet files for modeling

At this stage, I only want a simple chronological split pipeline.

# Importing libraries and Config

In [2]:
import pandas as pd
from pathlib import Path

## Env Variables setup

In [3]:
PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data"

INTERIM_DIR = DATA_DIR / "interim"
INPUT_PATH =  INTERIM_DIR / "transactions_validated.parquet"

OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = OUTPUT_DIR / "train.parquet"
TEST_FILE = OUTPUT_DIR / "test.parquet"
VAL_FILE = OUTPUT_DIR / "val.parquet"

print(f"Project directory: {PROJECT_DIR} | Exists: {PROJECT_DIR.exists()}")
print(f"Data directory: {DATA_DIR} | Exists: {DATA_DIR.exists()}")
print(f"Interim directory: {INTERIM_DIR} | Exists: {INTERIM_DIR.exists()}")
print(f"Input file: {INPUT_PATH} | Exists: {INPUT_PATH.exists()}")
print(f"Output directory: {OUTPUT_DIR} | Exists: {OUTPUT_DIR.exists()}")
print(f"Train file: {TRAIN_FILE} | Exists: {TRAIN_FILE.exists()}")
print(f"Test file: {TEST_FILE} | Exists: {TEST_FILE.exists()}")
print(f"Validation file: {VAL_FILE} | Exists: {VAL_FILE.exists()}")


Project directory: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | Exists: True
Data directory: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data | Exists: True
Interim directory: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\interim | Exists: True
Input file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\interim\transactions_validated.parquet | Exists: True
Output directory: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed | Exists: True
Train file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\train.parquet | Exists: False
Test file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\test.parquet | Exists: False
Validation file: C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\data\processed\val.parquet | Exists: False


# Load the Data and Basic Setup

In [5]:
df = pd.read_parquet(INPUT_PATH)
print("Loaded data shape:", df.shape)


Loaded data shape: (5078345, 12)


In [6]:
df.head()

,transaction_id,transaction_timestamp,from_bank_id,from_account_id,to_bank_id,to_account_id,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,txn_829ef002c65392e7_000,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,txn_074e10942a6f76c2_000,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,txn_558b19130bafb8d4_000,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,txn_a7af3dc87f0b38b0_000,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,txn_4eaa10cdc35a22e9_000,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 12 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         str           
 1   transaction_timestamp  datetime64[us]
 2   from_bank_id           str           
 3   from_account_id        str           
 4   to_bank_id             str           
 5   to_account_id          str           
 6   amount_received        float64       
 7   receiving_currency     str           
 8   amount_paid            float64       
 9   payment_currency       str           
 10  payment_format         str           
 11  is_laundering          int64         
dtypes: datetime64[us](1), float64(2), int64(1), str(8)
memory usage: 820.0 MB


In [8]:
print("Columns:", df.columns.tolist())

Columns: ['transaction_id', 'transaction_timestamp', 'from_bank_id', 'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received', 'receiving_currency', 'amount_paid', 'payment_currency', 'payment_format', 'is_laundering']


# check

## Sort chronologically

For a time-aware project like this, future rows should never appear in the training set before earlier rows.

In [11]:
df = df.sort_values(by=["transaction_timestamp","transaction_id"]).reset_index(drop=True)
df[["transaction_timestamp", "transaction_id"]].head()

,transaction_timestamp,transaction_id
0,2022-09-01,txn_000c8cd19174356a_000
1,2022-09-01,txn_0015c45f9cb45292_000
2,2022-09-01,txn_001dfc493364351d_000
3,2022-09-01,txn_0024335c600d3870_000
4,2022-09-01,txn_00346c68f75d61f6_000


In [12]:
print("Min Timestamp:", df["transaction_timestamp"].min())
print("Max Timestamp:", df["transaction_timestamp"].max())
print("Total Transactions:", len(df))

Min Timestamp: 2022-09-01 00:00:00
Max Timestamp: 2022-09-18 16:18:00
Total Transactions: 5078345


## Split Unique Timestamps
Instead of cutting by raw row number, I want to cut by unique timestamps. 
This avoids splitting the exact same timestamp across train/val/test, which is a cleaner first guard against temporal leakage.

split cutoffs - Simple 70 / 15 / 15 split.

In [19]:
#There is no need for dropna as there are no null values in the timestamp column but this is just to be on safe side. Same for sorting, we have already sorted the dataframe by timestamp and transaction_id but this is just to be on safe side.
unique_timestamps = pd.Series(df["transaction_timestamp"].dropna().sort_values().unique())
print("Number of unique timestamps:", len(unique_timestamps))
unique_timestamps.head()

Number of unique timestamps: 15018


0   2022-09-01 00:00:00
1   2022-09-01 00:01:00
2   2022-09-01 00:02:00
3   2022-09-01 00:03:00
4   2022-09-01 00:04:00
dtype: datetime64[us]

In [20]:
train_split = 0.70
test_split = 0.15
val_split = 0.15

assert abs(train_split + test_split + val_split - 1.0) < 1e-6, "Splits must sum to 1.0"

In [32]:
n_unique_tps = len(unique_timestamps)

train_end_idx = int(n_unique_tps * train_split) - 1
val_end_idx = int(n_unique_tps * (train_split + val_split)) - 1

train_cutoff_tp = unique_timestamps.iloc[train_end_idx]
val_cutoff_tp = unique_timestamps.iloc[val_end_idx]

print(f"Train cutoff timestamp: {train_cutoff_tp} (index {train_end_idx})")
print(f"Validation cutoff timestamp: {val_cutoff_tp} (index {val_end_idx})")


Train cutoff timestamp: 2022-09-08 07:11:00 (index 10511)
Validation cutoff timestamp: 2022-09-09 20:44:00 (index 12764)


In [34]:
train_df = df[df["transaction_timestamp"] <= train_cutoff_tp].copy()
val_df = df[(df["transaction_timestamp"] > train_cutoff_tp) & (df["transaction_timestamp"] <= val_cutoff_tp)].copy()
test_df = df[df["transaction_timestamp"] > val_cutoff_tp].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (3886017, 12)
Validation shape: (896971, 12)
Test shape: (295357, 12)


## Chronology checks

In [35]:
print("Train time range:")
print(train_df["transaction_timestamp"].min(), "->", train_df["transaction_timestamp"].max())

print("\nValidation time range:")
print(val_df["transaction_timestamp"].min(), "->", val_df["transaction_timestamp"].max())

print("\nTest time range:")
print(test_df["transaction_timestamp"].min(), "->", test_df["transaction_timestamp"].max())

Train time range:
2022-09-01 00:00:00 -> 2022-09-08 07:11:00

Validation time range:
2022-09-08 07:12:00 -> 2022-09-09 20:44:00

Test time range:
2022-09-09 20:45:00 -> 2022-09-18 16:18:00


In [37]:
assert train_df["transaction_timestamp"].max() <= train_cutoff_tp
assert val_df["transaction_timestamp"].min() > train_cutoff_tp

assert val_df["transaction_timestamp"].max() <= val_cutoff_tp
assert test_df["transaction_timestamp"].min() > val_cutoff_tp

print("Time boundary checks passed.")

Time boundary checks passed.


In [38]:
total_split_rows = len(train_df) + len(val_df) + len(test_df)

print("Original rows:", len(df))
print("Rows across splits:", total_split_rows)

assert total_split_rows == len(df)
print("Row conservation check passed.")

Original rows: 5078345
Rows across splits: 5078345
Row conservation check passed.


In [39]:
all_ids = pd.concat([
    train_df["transaction_id"],
    val_df["transaction_id"],
    test_df["transaction_id"]
], axis=0)

print("Combined ID count:", len(all_ids))
print("Unique combined ID count:", all_ids.nunique())

assert len(all_ids) == all_ids.nunique()
print("No duplicate transaction_id across splits.")

Combined ID count: 5078345
Unique combined ID count: 5078345
No duplicate transaction_id across splits.


# Check class balance in each split

In [40]:
def split_summary(name: str, split_df: pd.DataFrame) -> pd.DataFrame:
    total = len(split_df)
    target_counts = split_df["is_laundering"].value_counts(dropna= False).sort_index()

    summary = target_counts.to_frame("count")
    summary["pct"] = (summary["count"] /total *100).round(4)

    print(f"\n{name} Summary:")
    print("-" * 40)
    print("Rows:",total)
    print("Start:",split_df["transaction_timestamp"].min())
    print("End:",split_df["transaction_timestamp"].max())
    print("Summary:\n",summary)
    print("-" * 40)

    return summary

In [42]:
train_summary = split_summary("Train:",train_df)
val_summary = split_summary("Validation:",val_df)
test_summary = split_summary("Test",test_df)


Train: Summary:
----------------------------------------
Rows: 3886017
Start: 2022-09-01 00:00:00
End: 2022-09-08 07:11:00
Summary:
                  count      pct
is_laundering                  
0              3882877  99.9192
1                 3140   0.0808
----------------------------------------

Validation: Summary:
----------------------------------------
Rows: 896971
Start: 2022-09-08 07:12:00
End: 2022-09-09 20:44:00
Summary:
                 count      pct
is_laundering                 
0              896093  99.9021
1                 878   0.0979
----------------------------------------

Test Summary:
----------------------------------------
Rows: 295357
Start: 2022-09-09 20:45:00
End: 2022-09-18 16:18:00
Summary:
                 count      pct
is_laundering                 
0              294198  99.6076
1                1159   0.3924
----------------------------------------


In [43]:
split_sizes = pd.Series({
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
})

(split_sizes / len(df) * 100).round(2)

train_rows    76.52
val_rows      17.66
test_rows      5.82
dtype: float64

# End

**Split observation:**
- This split was created using ordered unique timestamps to avoid placing the same timestamp in multiple splits.
- As a result, row percentages are not exactly 70/15/15, because transaction volume is uneven across timestamps.
- The split is still chronologically correct, but the later test window has a noticeably higher positive rate than the training window, which suggests temporal distribution shift.

When I do the baseline model, I should be extra careful to follow the project’s evaluation policy:
- do not focus on accuracy
- use precision, recall, F1, PR-AUC, ROC-AUC
- do not lock the threshold at 0.50
- choose threshold from validation data, not test data